In [ ]:
%%capture

import subprocess
# reference: see 8b in https://irsa.ipac.caltech.edu/data/ZTF/docs/releases/ztf_release_notes_latest
ref_names = [
    'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000829/zr/ccd10/q4/ztf_000829_zr_c10_q4_refimg.fits',
    'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000626/zr/ccd12/q2/ztf_000626_zr_c12_q2_refimg.fits'
    # can also add zg or other band ref images if needed.
]
for name in ref_names:
    result = subprocess.run(['wget', f'{name}'])#, stdout=subprocess.DEVNULL);
    print(result)

from astropy.io import fits
fits.open('/content/ztf_000626_zr_c12_q2_refimg.fits')[0].header['MAGLIM'], fits.open('/content/ztf_000829_zr_c10_q4_refimg.fits')[0].header['MAGLIM']

In [ ]:
!apt-get update
!pip install latex
!sudo apt install texlive texlive-latex-extra texlive-fonts-recommended dvipng
!sudo apt install cm-super

In [ ]:
!pip install scienceplots

In [ ]:
!mkdir /root/.kaggle
!mv kaggle.json /root/.kaggle
!chmod 600 /root/.kaggle/kaggle.json
!kaggle kernels output yashg1002/ztf-deconvolution-run -p /content/ztf_deconv_results

In [ ]:
# The files in this zip file are obtained from Kaggle
!unzip /content/deepref_experiment.zip

In [ ]:
import os
import numpy as np
import pandas as pd
import glob
from astropy.io import fits
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

import subprocess

from astropy.io import fits
from astropy.visualization import MinMaxInterval, ImageNormalize, SqrtStretch, ZScaleInterval, PercentileInterval, LogStretch

from scipy.stats import binned_statistic
from scipy.stats import median_abs_deviation

from IPython.display import Image

In [ ]:
# modified cut conditions for ref-sci comparison. basically we set min mag to 24 for deepref and deconv.

def cut_condition(cat):
    return (cat['MAG_ISO'] >= 15) & (cat['MAG_ISO'] <= 24) & (cat['FWHM_IMAGE'] > 0) & (cat['FWHM_IMAGE'] <= 4) & (cat['ELLIPTICITY'] < 0.5)

def dcut_condition(cat):
    return (cat['MAG_ISO'] <= 24) & (cat['FWHM_IMAGE'] > 0) & (cat['FWHM_IMAGE'] <= 4)

def cut_condition2(cat):
    return (cat['MAG_ISO_2'] >= 15) & (cat['MAG_ISO_2'] <= 24) & (cat['FWHM_IMAGE_2'] > 0) & (cat['FWHM_IMAGE_2'] <= 4) & (cat['ELLIPTICITY_2'] < 0.5) & (cat['MAG_ISO_1'] <= 24) & (cat['FWHM_IMAGE_1'] > 0) & (cat['FWHM_IMAGE_1'] <= 4)

def check_particular_flag(x, nth_bit_from_right=0):
    """Checks whether a particular FLAG corresponding is set or not.
    nth_bit_from_right = 0 means the rightmost bit (0th bit).
    """
    x = int(x)
    return x & (1<<nth_bit_from_right)  # nth_bit_from_right=1 corresponds to 2^1=2.

# NOTE: We also remove sources with FLAGS > 7.
def clean_catalog_using_flags_condition(cat, colname='FLAGS'):
    # cond1 = cat[colname].astype(int).apply(func=check_particular_flag, nth_bit_from_right=4).astype(bool)
    # cond2 = cat[colname].astype(int).apply(func=check_particular_flag, nth_bit_from_right=5).astype(bool)
    # cond3 = cat[colname].astype(int).apply(func=check_particular_flag, nth_bit_from_right=6).astype(bool)
    # cond4 = cat[colname].astype(int).apply(func=check_particular_flag, nth_bit_from_right=7).astype(bool)
    # return (~cond1) & (~cond2) & (~cond3) & (~cond4)
    return cat[colname] <= 7

# def cut_condition2_one_to_one(cat):
#     assert cat['MAG_ISO_2_x'].equals(cat['MAG_ISO_2_y'])
#     assert cat['FWHM_IMAGE_2_x'].equals(cat['FWHM_IMAGE_2_y'])
#     assert cat['ELLIPTICITY_2_x'].equals(cat['ELLIPTICITY_2_y'])
#     return (cat['MAG_ISO_2_x'] >= 15) & (cat['MAG_ISO_2_x'] <= 20.5) & (cat['FWHM_IMAGE_2_x'] >= 0) & (cat['FWHM_IMAGE_2_x'] <= 4) & (cat['ELLIPTICITY_2_x'] < 0.5)

In [ ]:
%%capture
import subprocess
# reference: see 8b in https://irsa.ipac.caltech.edu/data/ZTF/docs/releases/ztf_release_notes_latest
ref_names = [
    # 'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000829/zr/ccd10/q4/ztf_000829_zr_c10_q4_refimg.fits',
    'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000626/zr/ccd12/q2/ztf_000626_zr_c12_q2_refimg.fits',
    'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000619/zr/ccd11/q2/ztf_000619_zr_c11_q2_refimg.fits',
    'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000635/zr/ccd12/q4/ztf_000635_zr_c12_q4_refimg.fits',
    # 'https://irsa.ipac.caltech.edu/ibe/data/ztf/products/deep/000/field000251/zr/ccd16/q4/ztf_000251_zr_c16_q4_refimg.fits'
    # can also add zg or other band ref images if needed.
]
for name in ref_names:
    result = subprocess.run(['wget', f'{name}'])#, stdout=subprocess.DEVNULL);
    print(result)

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd

directory = 'final_catalogs_refsci'
if os.path.exists(directory):
    shutil.rmtree(directory)
os.makedirs(directory)

DIRNAME = '/content'

###################################################################
# ID = '251'
# CROSSMATCH_PREFIX = 'ztf_000251_zr_c16_q4_refimg.fits'
# maglim_sci = 20.11  # this is for 251-r. It will be 21.39 for 828-r.

# SOURCE_CATALOGS = [
#     '/content/ref_ztf_000251_zr_c16_q4_refimg.fits_scat_sextractor.csv',
#     '/content/deconv_ztf_20180831500891_000251_zr_c16_o_q4_sciimg.fits_scat_sextractor.csv'
# ]

# ---------------------------------------------
# ID = '619'
# CROSSMATCH_PREFIX = 'ztf_000619_zr_c11_q2_refimg.fits'
# maglim_sci = 21.32  # this is for 619-r. It will be 21.39 for 828-r.

# SOURCE_CATALOGS = [
#     '/content/ref_ztf_000619_zr_c11_q2_refimg.fits_scat_sextractor.csv',
#     '/content/deconv_ztf_20230127346250_000619_zr_c11_o_q2_sciimg.fits_scat_sextractor.csv'
# ]

# ---------------------------------------------
ID = '635'
CROSSMATCH_PREFIX = 'ztf_000635_zr_c12_q4_refimg.fits'
maglim_sci = 20.94  # this is for 635-r. It will be 21.39 for 828-r.

SOURCE_CATALOGS = [
    '/content/ref_ztf_000635_zr_c12_q4_refimg.fits_scat_sextractor.csv',
    '/content/deconv_ztf_20231021119745_000635_zr_c12_o_q4_sciimg.fits_scat_sextractor.csv'
]

# ---------------------------------------------
# ID = '626'
# CROSSMATCH_PREFIX = 'ztf_000626_zr_c12_q2_refimg.fits'
# maglim_sci = 20.27  # this is for 626-r. It will be 21.39 for 828-r.

# SOURCE_CATALOGS = [
#     '/content/ref_ztf_000626_zr_c12_q2_refimg.fits_scat_sextractor.csv',
#     '/content/deconv_ztf_20221120493183_000626_zr_c12_o_q2_sciimg.fits_scat_sextractor.csv'
# ]

# ---------------------------------------------

# ID = '829'
# CROSSMATCH_PREFIX = 'ztf_000829_zr_c10_q4_refimg.fits'
# maglim_sci = 21.39  # 20.27 is for 626-r. It will be 21.39 for 828-r.

# SOURCE_CATALOGS = [
#     '/content/ref_ztf_000829_zr_c10_q4_refimg.fits_scat_sextractor.csv',
#     '/content/deconv_ztf_20220706370127_000829_zr_c10_o_q4_sciimg.fits_scat_sextractor.csv'
# ]
###################################################################

title = rf'{ID}-$r$'
prefix = f'/{ID}'
maglim_ref = fits.open(f'/content/{CROSSMATCH_PREFIX}')[0].header['MAGLIM']

IMG_NAMES = [
    f'/content/{CROSSMATCH_PREFIX}',
]

# Remember: In crossmatched catalogs, 1 is deconvolved and 2 means original.
# Topcat settings: 2-d cartesian, 1.383 error, and X_IMAGE_DBL and Y_IMAGE_DBL are used. Equivalent _WORLD values used for ref vs deconv crossmatching.
CROSSMATCHED_CATALOG_best_match_for_each_table2_row_1_and_2 = f'{CROSSMATCH_PREFIX}_crossmatched_best_match_for_each_table2_row_1_and_2_stilts_ref.csv'
CROSSMATCHED_CATALOG_best_match_for_each_table1_row_1_and_2 = f'{CROSSMATCH_PREFIX}_crossmatched_best_match_for_each_table1_row_1_and_2_stilts_ref.csv'
CROSSMATCHED_CATALOG_best_match_symmetric_1_and_2 = f'{CROSSMATCH_PREFIX}_crossmatched_best_match_symmetric_1_and_2_stilts_ref.csv'
CROSSMATCHED_CATALOG_all_matches = f'{CROSSMATCH_PREFIX}_crossmatched_all_matches_1_and_2_stilts_ref.csv'

CROSSMATCHED_CATALOGS = [
    CROSSMATCHED_CATALOG_best_match_for_each_table2_row_1_and_2,
    CROSSMATCHED_CATALOG_best_match_symmetric_1_and_2,
    CROSSMATCHED_CATALOG_best_match_for_each_table1_row_1_and_2,
    CROSSMATCHED_CATALOG_all_matches
]
CROSSMATCHED_CATALOGS_NAMES = [
    'best_match_for_each_table2_row_1_and_2',
    'best_match_symmetric_1_and_2',
    'best_match_for_each_table1_row_1_and_2',
    'all_matches_1_and_2'
]

CROSSMATCHED_CATALOGS_2_not_1 = f'{CROSSMATCH_PREFIX}_crossmatched_best_match_for_each_table2_row_2_not_1_stilts_ref.csv'
CROSSMATCHED_CATALOGS_1_not_2 = f'{CROSSMATCH_PREFIX}_crossmatched_best_match_for_each_table1_row_1_not_2_stilts_ref.csv'

IMAGE_FITS_NAMES = [
    f'ref_subdiv_{CROSSMATCH_PREFIX}', f'deconvolved_subdiv_{CROSSMATCH_PREFIX}'
]

NAMES = [CROSSMATCH_PREFIX] * len(CROSSMATCHED_CATALOGS)

catname = CROSSMATCHED_CATALOGS
mcat1 = pd.read_csv(os.path.join(DIRNAME, catname[0]))  # best_match_for_each_table2_row_1_and_2 (Each row from table 2 will appear a maximum of once in the result, but rows from table 1 may appear multiple times.)
mcat2 = pd.read_csv(os.path.join(DIRNAME, catname[1]))  # best_match_symmetric_1_and_2
mcat3 = pd.read_csv(os.path.join(DIRNAME, catname[2]))  # best_match_for_each_table1_row_1_and_2 (Each row from table 1 will appear a maximum of once in the result, but rows from table 2 may appear multiple times.)
# mcat4 is all matches, but that is not used for the plot here.
mcat4 = pd.read_csv(os.path.join(DIRNAME, catname[3]))  # all_matches

catname = [CROSSMATCHED_CATALOGS_2_not_1, CROSSMATCHED_CATALOGS_1_not_2, CROSSMATCHED_CATALOGS[2], SOURCE_CATALOGS]
um_ocat = pd.read_csv(os.path.join(DIRNAME, catname[0]))
um_dcat = pd.read_csv(os.path.join(DIRNAME, catname[1]))
ocat = pd.read_csv(os.path.join(DIRNAME, catname[3][0]))
dcat = pd.read_csv(os.path.join(DIRNAME, catname[3][1]))

many_to_one = []
one_to_one1 = []
for mcat3_col1_2 in mcat3['col1_2'].unique():
    mcat3_subset = mcat3[mcat3['col1_2'] == mcat3_col1_2]
    if len(mcat3_subset) == 1:
        one_to_one1.append(mcat3_subset)
    elif len(mcat3_subset) > 1:
        many_to_one.append(mcat3_subset)
        # one_to_one1.append(mcat3_subset[mcat3_subset['Separation'] == mcat3_subset['Separation'].min()])

one_to_many = []
one_to_one2 = []
for mcat1_col1_1 in mcat1['col1_1'].unique():
    mcat1_subset = mcat1[mcat1['col1_1'] == mcat1_col1_1]
    if len(mcat1_subset) == 1:
        one_to_one2.append(mcat1_subset)
    elif len(mcat1_subset) > 1:
        one_to_many.append(mcat1_subset)
        # one_to_one2.append(mcat1_subset[mcat1_subset['Separation'] == mcat1_subset['Separation'].min()])

# one_to_one = pd.merge(pd.concat((one_to_one1)), pd.concat((one_to_one2)), how='inner', on=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2', 'X_IMAGE_DBL_1', 'Y_IMAGE_DBL_1'])

one_to_one1 = pd.concat((one_to_one1))
one_to_one2 = pd.concat((one_to_one2))
# NOTE: One can use the merge option to create one to one as well, but perhaps the concat followed by removal of duplicates is better suited.
# one_to_one = pd.merge(one_to_one1, one_to_one2, how='inner', on=['col1_2', 'col1_1'])
one_to_one = pd.concat((one_to_one1, one_to_one2)).drop_duplicates(subset=['col1_2', 'col1_1'])

# assert len(one_to_one) == len(mcat2)  # NOTE: This wouldn't be equal to mcat2 unless we uncomment the below two lines above:
"""
# one_to_one1.append(mcat3_subset[mcat3_subset['Separation'] == mcat3_subset['Separation'].min()])
# one_to_one2.append(mcat1_subset[mcat1_subset['Separation'] == mcat1_subset['Separation'].min()])
"""

# many_to_one = pd.DataFrame(np.vstack((many_to_one)))
# many_to_one.columns = mcat3.columns
# one_to_many = pd.DataFrame(np.vstack((one_to_many)))
# one_to_many.columns = mcat1.columns

all_zero_dataframe_indicator_many_to_one = False
all_zero_dataframe_indicator_one_to_many = False

if len(many_to_one) > 0:
    many_to_one = pd.DataFrame(np.vstack((many_to_one)))
    many_to_one.columns = mcat3.columns
else:
    _dummy = np.zeros_like(mcat3)
    # _dummy[:] = np.nan
    many_to_one = pd.DataFrame(_dummy)
    many_to_one.columns = mcat3.columns
    all_zero_dataframe_indicator_many_to_one = True
if len(one_to_many) > 0:
    one_to_many = pd.DataFrame(np.vstack((one_to_many)))
    one_to_many.columns = mcat1.columns
else:
    _dummy = np.zeros_like(mcat1)
    # _dummy[:] = np.nan
    one_to_many = pd.DataFrame(_dummy)
    one_to_many.columns = mcat1.columns
    all_zero_dataframe_indicator_one_to_many = True

# many_to_one = many_to_one.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2'])
# one_to_many = one_to_many.drop_duplicates(subset=['X_IMAGE_DBL_1', 'Y_IMAGE_DBL_1'])

# Initialize the 1-many and many-1 cut dataframes.
many_to_one_cut = many_to_one.copy(deep=True)
one_to_many_cut = one_to_many.copy(deep=True)

# For one-to-many and many-to-one, we cannot simply apply the cut conditions since post that, this could become one-to-one.
# Note that those that are converted from 1-many/many-1 to 1-1 are still not included in 1-1 for simplicity.
# So we need a slightly modified machinery.
# First for many-to-one.
previously_many_to_one_but_not_now = 0
previously_one_to_many_but_not_now = 0
for m2o_col1_2 in many_to_one['col1_2'].unique():
    many_to_one_subset = many_to_one[many_to_one['col1_2'] == m2o_col1_2]
    mosc = many_to_one_subset[cut_condition2(many_to_one_subset)]
    if len(mosc) < len(many_to_one_subset):  # Then this means one of the matches was removed, so we remove the entire many-1 match.
        # assert one_to_one.shape[1] == many_to_one_subset_cut.shape[1]  # No. of columns must be same.
        # Add it to one-to-one.
        # one_to_one = pd.concat((one_to_one, many_to_one_subset_cut))
        # and remove from many-to-one. But don't directly remove from many_to_one but instead remove from many_to_one_cut.
        many_to_one_cut = many_to_one_cut[many_to_one_cut['col1_2'] != m2o_col1_2]
        previously_many_to_one_but_not_now = previously_many_to_one_but_not_now + 1
# Now for one-to-many.
for o2m_col1_1 in one_to_many['col1_1'].unique():
    one_to_many_subset = one_to_many[one_to_many['col1_1'] == o2m_col1_1]
    omsc = one_to_many_subset[cut_condition2(one_to_many_subset)]
    if len(omsc) < len(one_to_many_subset):  # Then this means one of the matches was removed, so we remove the entire 1-many match.
        # assert one_to_one.shape[1] == one_to_many_subset_cut.shape[1]  # No. of columns must be same.
        # Add it to one-to-one.
        # one_to_one = pd.concat((one_to_one, one_to_many_subset_cut))
        # and remove from one-to-many. But don't directly remove from one_to_many but instead remove from one_to_many_cut.
        one_to_many_cut = one_to_many_cut[one_to_many_cut['col1_1'] != o2m_col1_1]
        previously_one_to_many_but_not_now = previously_one_to_many_but_not_now + 1

ocat_cut = ocat[cut_condition(ocat)]
ocat_cut = ocat_cut[clean_catalog_using_flags_condition(ocat_cut)]
dcat_cut = dcat[dcut_condition(dcat)]
one_to_one_cut = one_to_one[(cut_condition2(one_to_one))]
one_to_one_cut = one_to_one_cut[clean_catalog_using_flags_condition(one_to_one_cut, colname='FLAGS_2')]
# Note that we cannot simply apply the cut conditions as below. See above.
# many_to_one_cut = many_to_one[cut_condition2(many_to_one)]
# one_to_many_cut = one_to_many[cut_condition2(one_to_many)]
unmatches_original_cut = um_ocat[cut_condition(um_ocat)]; unmatches_original_cut = unmatches_original_cut[clean_catalog_using_flags_condition(unmatches_original_cut)]
unmatches_deconvolved_cut = um_dcat[dcut_condition(um_dcat)]

# We can simply remove based on FLAGS_2 only for many_to_one_cut but not for one_to_many. So we have another for loop below.
many_to_one_cut = many_to_one_cut[clean_catalog_using_flags_condition(many_to_one_cut, colname='FLAGS_2')]
one_to_many_cut = one_to_many_cut[clean_catalog_using_flags_condition(one_to_many_cut, colname='FLAGS_2')]
for o2m_col1_1 in one_to_many_cut['col1_1'].unique():
    one_to_many_cut_subset = one_to_many_cut[one_to_many_cut['col1_1'] == o2m_col1_1]
    if (len(one_to_many_cut_subset) % 2) != 0:  # means at least one orig source is removed due to the FLAGS condition, so remove all instances.
        one_to_many_cut = one_to_many_cut[one_to_many_cut['col1_1'] != o2m_col1_1]
        previously_one_to_many_but_not_now = previously_one_to_many_but_not_now + 1

# x = pd.merge(
#     many_to_one.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     one_to_one.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     on=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']

# )
# y = pd.merge(
#     many_to_one.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     one_to_many.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     on=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']

# )
# z = pd.merge(
#     one_to_many.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     one_to_one.drop_duplicates(subset=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']),
#     on=['X_IMAGE_DBL_2', 'Y_IMAGE_DBL_2']

# )

# assert one_to_one['col1_2_x'].equals(one_to_one['col1_2_y'])

x2 = pd.merge(
    many_to_one.drop_duplicates(subset=['col1_2']),
    one_to_one.drop_duplicates(subset=['col1_2']),#.rename(columns={'col1_2_x': 'col1_2'}),
    on=['col1_2']
)
y2 = pd.merge(
    many_to_one.drop_duplicates(subset=['col1_2']),
    one_to_many.drop_duplicates(subset=['col1_2']),
    on=['col1_2']
)
z2 = pd.merge(
    one_to_many.drop_duplicates(subset=['col1_2']),
    one_to_one.drop_duplicates(subset=['col1_2']),#.rename(columns={'col1_2_x': 'col1_2'}),
    on=['col1_2']
)
x1 = pd.merge(
    many_to_one.drop_duplicates(subset=['col1_1']),
    one_to_one.drop_duplicates(subset=['col1_1']),#.rename(columns={'col1_2_x': 'col1_2'}),
    on=['col1_1']
)
y1 = pd.merge(
    many_to_one.drop_duplicates(subset=['col1_1']),
    one_to_many.drop_duplicates(subset=['col1_1']),
    on=['col1_1']
)
z1 = pd.merge(
    one_to_many.drop_duplicates(subset=['col1_1']),
    one_to_one.drop_duplicates(subset=['col1_1']),#.rename(columns={'col1_2_x': 'col1_2'}),
    on=['col1_1']
)
# xyz = pd.merge(
#     one_to_many.drop_duplicates(subset=['col1_2']),
#     one_to_one.drop_duplicates(subset=['col1_2']),#.rename(columns={'col1_2_x': 'col1_2'}),
#     on=['col1_2']
# ).merge(
#     one_to_one.drop_duplicates(subset=['col1_2']),
#     on=['col1_2']
# )

w1 = pd.merge(
    one_to_many.drop_duplicates(subset=['col1_2']),
    unmatches_original_cut.drop_duplicates(subset=['col1']).rename(columns={'X_IMAGE_DBL': 'X_IMAGE_DBL_2', 'Y_IMAGE_DBL': 'Y_IMAGE_DBL_2', 'col1': 'col1_2'}),
    on=['col1_2']
)
w2 = pd.merge(
    one_to_one.drop_duplicates(subset=['col1_2']),
    unmatches_original_cut.drop_duplicates(subset=['col1']).rename(columns={'X_IMAGE_DBL': 'X_IMAGE_DBL_2', 'Y_IMAGE_DBL': 'Y_IMAGE_DBL_2', 'col1': 'col1_2'}),
    on=['col1_2']
)
w3 = pd.merge(
    many_to_one.drop_duplicates(subset=['col1_2']),
    unmatches_original_cut.drop_duplicates(subset=['col1']).rename(columns={'X_IMAGE_DBL': 'X_IMAGE_DBL_2', 'Y_IMAGE_DBL': 'Y_IMAGE_DBL_2', 'col1': 'col1_2'}),
    on=['col1_2']
)
assert len(w1) == 0
assert len(w2) == 0
assert len(w3) == 0

if not len(ocat) == len(one_to_one) + len(um_ocat) + len(many_to_one['col1_2'].unique()) + len(one_to_many['col1_2'].unique()) - (len(x2) + len(y2) + len(z2)):
    print(f"ocat assertion error for {title}\t{len(ocat)} vs. {len(one_to_one) + len(um_ocat) + len(many_to_one['col1_2'].unique()) + len(one_to_many['col1_2'].unique()) - (len(x2) + len(y2) + len(z2))}")
if not len(dcat) == len(one_to_one) + len(um_dcat) + len(many_to_one['col1_1'].unique()) + len(one_to_many['col1_1'].unique()) - (len(x1) + len(y1) + len(z1)):
    print(f"dcat assertion error for {title}\t{len(dcat)} vs. {len(one_to_one) + len(um_dcat) + len(many_to_one['col1_1'].unique()) + len(one_to_many['col1_1'].unique()) - (len(x1) + len(y1) + len(z1))}")
# print(len(xyz), len(x), len(y), len(z), len(many_to_one['col1_2'].unique()), len(one_to_many['col1_2'].unique()), len(um_ocat['col1'].unique()), len(w1), len(w2), len(w3))

############### Assertion for one to many and many to one matches ###############
for m2o_col1_2 in many_to_one['col1_2'].unique():
    many_to_one_subset = many_to_one[many_to_one['col1_2'] == m2o_col1_2]
    assert len(many_to_one_subset) > 1

for o2m_col1_1 in one_to_many['col1_1'].unique():
    one_to_many_subset = one_to_many[one_to_many['col1_1'] == o2m_col1_1]
    assert len(one_to_many_subset) > 1
#################################################################################
############### Assertion for one to one matches ###############
assert len(one_to_one['col1_1']) == len(one_to_one['col1_1'].unique())
assert len(one_to_one['col1_2']) == len(one_to_one['col1_2'].unique())
################################################################

oflux = ocat["FLUX_ISO"].sum()
dflux = dcat["FLUX_ISO"].sum()
# oflux_naive = np.sum(fits.getdata(IMG_NAMES[0]) - fits.getdata(IMG_NAMES[0].replace('orig_subdiv', 'orig_bkg')))
# Note that we subtract the original bkg in the below line for dflux_naive because this assumes --add_bkg_to_deconvolved was used when exectuing `run.py`.
# If --add_bkg_to_deconvolved was not mentioned, only the following must be kept:
# ```
# np.sum(fits.getdata(IMG_NAMES[0].replace('orig_subdiv', 'deconvolved_subdiv')))
#```
# dflux_naive = np.sum(fits.getdata(IMG_NAMES[0].replace('orig_subdiv', 'deconvolved_subdiv')))
oflux_naive, dflux_naive = np.nan, np.nan
print(f'Sum of flxues of all original and deconvolved sources (respectively): {oflux} and {dflux}, d - o = {dflux-oflux}, error = {100*(1-(dflux/oflux))}%')
print(f'Naive flux estimates for original and deconvolved (respectively): {oflux_naive}, {dflux_naive}')

# s1, edges1, _ = binned_statistic(mo, mo - md, statistic=median_abs_deviation, bins=np.linspace(0, max(mo), 40))
# best_deconv_mag = edges1[1:][np.where(s1 < 0.01)[0][-1]]
# print(f'The faintest orig source with reasonable flux conservation (scatter < 0.01 mag) has magnitude = {best_deconv_mag}')
# best_deconv_mag = edges1[1:][np.where(s1 < 0.1)[0][-1]]
# print(f'The faintest orig source with reasonable flux conservation (scatter < 0.1 mag) has magnitude = {best_deconv_mag}')

#################################################################################

#################### SAVE CATALOGS #############################################################################
ocat.to_csv(os.path.join(directory, 'ref' + prefix.split('/')[-1] + '.csv'))
dcat.to_csv(os.path.join(directory, 'deconvolved' + prefix.split('/')[-1] + '.csv'))
one_to_one.to_csv(os.path.join(directory, 'one_to_one' + prefix.split('/')[-1] + '.csv'))
many_to_one.to_csv(os.path.join(directory, 'many_to_one' + prefix.split('/')[-1] + '.csv'))
one_to_many.to_csv(os.path.join(directory, 'one_to_many' + prefix.split('/')[-1] + '.csv'))
um_ocat.to_csv(os.path.join(directory, 'unmatches_original' + prefix.split('/')[-1] + '.csv'))
um_dcat.to_csv(os.path.join(directory, 'unmatches_deconvolved' + prefix.split('/')[-1] + '.csv'))

ocat_cut.to_csv(os.path.join(directory, 'ref_cut_' + prefix.split('/')[-1] + '.csv'))
dcat_cut.to_csv(os.path.join(directory, 'deconvolved_cut_' + prefix.split('/')[-1] + '.csv'))
one_to_one_cut.to_csv(os.path.join(directory, 'one_to_one_cut_' + prefix.split('/')[-1] + '.csv'))
many_to_one_cut.to_csv(os.path.join(directory, 'many_to_one_cut_' + prefix.split('/')[-1] + '.csv'))
one_to_many_cut.to_csv(os.path.join(directory, 'one_to_many_cut_' + prefix.split('/')[-1] + '.csv'))
unmatches_original_cut.to_csv(os.path.join(directory, 'unmatches_original_cut_' + prefix.split('/')[-1] + '.csv'))
unmatches_deconvolved_cut.to_csv(os.path.join(directory, 'unmatches_deconvolved_cut_' + prefix.split('/')[-1] + '.csv'))
################################################################################################################

print(
    f"{prefix.split('/')[-1]}, {len(ocat)} ({len(ocat_cut)}), {len(dcat)} ({len(dcat_cut)}), {len(one_to_one)} ({len(one_to_one_cut)}), {len(many_to_one)} ({len(many_to_one_cut)}), {len(one_to_many)} ({len(one_to_many_cut)}), {len(um_ocat)} ({len(unmatches_original_cut)}), {len(um_dcat)} ({len(unmatches_deconvolved_cut)})"
)
one_to_many_vc = one_to_many_cut['col1_1'].value_counts()
print(f'1-many: two matches = {len(one_to_many_vc[one_to_many_vc == 2])}, three matches = {len(one_to_many_vc[one_to_many_vc == 3])}, four matches = {len(one_to_many_vc[one_to_many_vc == 4])}')
many_to_one_vc = many_to_one_cut['col1_2'].value_counts()
print(f'many-1: two matches = {len(many_to_one_vc[many_to_one_vc == 2])}, three matches = {len(many_to_one_vc[many_to_one_vc == 3])}, four matches = {len(many_to_one_vc[many_to_one_vc == 4])}')

In [ ]:
ocat_cut = pd.read_csv(os.path.join(directory, 'ref_cut_' + prefix.split('/')[-1] + '.csv'))
dcat_cut = pd.read_csv(os.path.join(directory, 'deconvolved_cut_' + prefix.split('/')[-1] + '.csv'))
one_to_one_cut = pd.read_csv(os.path.join(directory, 'one_to_one_cut_' + prefix.split('/')[-1] + '.csv'))

ocat = pd.read_csv(os.path.join(directory, 'ref' + prefix.split('/')[-1] + '.csv'))
dcat = pd.read_csv(os.path.join(directory, 'deconvolved' + prefix.split('/')[-1] + '.csv'))
one_to_one = pd.read_csv(os.path.join(directory, 'one_to_one' + prefix.split('/')[-1] + '.csv'))

In [ ]:
print(one_to_one_cut.shape)
print(one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > maglim_sci) & (one_to_one_cut['MAG_ISO_1'] <= 21.5)].shape)
print(dcat_cut[(dcat_cut['MAG_ISO'] > maglim_sci) & (dcat_cut['MAG_ISO'] <= 21.5)].shape)
# print(ocat_cut[(ocat_cut['MAG_ISO'] > maglim_sci) & (ocat_cut['MAG_ISO'] <= 21.5)].shape)
print(ocat_cut.shape)

In [ ]:
print(one_to_one_cut.shape)
print(one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > 21.5) & (one_to_one_cut['MAG_ISO_1'] <= 24)].shape)
print(dcat_cut[(dcat_cut['MAG_ISO'] > 21.5) & (dcat_cut['MAG_ISO'] <= 24)].shape)
# print(ocat_cut[(ocat_cut['MAG_ISO'] > maglim_sci) & (ocat_cut['MAG_ISO'] <= 21.5)].shape)
print(ocat_cut.shape)

In [ ]:
one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > 21.5) & (one_to_one_cut['MAG_ISO_1'] <= 24)]['MAG_ISO_1'].hist()

In [ ]:
ocat.shape, ocat_cut.shape, dcat.shape, dcat_cut.shape

In [ ]:
# after first review.
unmatch_dcat = pd.read_csv(os.path.join(directory, 'unmatches_deconvolved_cut_' + prefix.split('/')[-1] + '.csv'))
one_to_one_cut = pd.read_csv(os.path.join(directory, 'one_to_one_cut_' + prefix.split('/')[-1] + '.csv'))
ocat_cut = pd.read_csv(os.path.join(directory, 'ref_cut_' + prefix.split('/')[-1] + '.csv'))
dcat_cut = pd.read_csv(os.path.join(directory, 'deconvolved_cut_' + prefix.split('/')[-1] + '.csv'))
ocat = pd.read_csv(os.path.join(directory, 'ref' + prefix.split('/')[-1] + '.csv'))
dcat = pd.read_csv(os.path.join(directory, 'deconvolved' + prefix.split('/')[-1] + '.csv'))

print(f'Median FWHM and ellipticity of one-to-one matches = {np.median(one_to_one_cut["FWHM_IMAGE_1"]), np.median(one_to_one_cut["ELLIPTICITY_1"])}')
print(f'Scatter about median of FWHM and ellipticity of one-to-one matches = {median_abs_deviation(one_to_one_cut["FWHM_IMAGE_1"]), median_abs_deviation(one_to_one_cut["ELLIPTICITY_1"])}')

# see https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=terms-modified-z-score
robust_zscores = (unmatch_dcat['FWHM_IMAGE'] - np.median(one_to_one_cut["FWHM_IMAGE_1"])) / (1.4826 * median_abs_deviation(one_to_one_cut["FWHM_IMAGE_1"]))
robust_zscores_only = robust_zscores[(robust_zscores < -3.5) | (robust_zscores > 3.5)]
print(f'No. of unreliable deconv sources using zscore criteria = {len(robust_zscores_only)}/{len(unmatch_dcat)}')
# robust_zscores_fwhmzero = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatch_dcat['FWHM_IMAGE'], 0.)))]
# print(f'No. of unreliable deconv sources using zscore criteria and fwhm near zero criteria = {len(robust_zscores_fwhmzero)}/{len(unmatch_dcat)}')
# unmatch_dcat_astrophysical_final_first = unmatch_dcat[(robust_zscores >= -3) & (robust_zscores <= 3)]
# unmatch_dcat_astrophysical_final = unmatch_dcat_astrophysical_final_first[unmatch_dcat_astrophysical_final_first['ELLIPTICITY'] < 0.8]
# print(f'Final no. of astrophysical sources = {len(unmatch_dcat_astrophysical_final)}')

# after first review END.

unmatch_dcat_astrophysical = unmatch_dcat[(unmatch_dcat['FLAGS'] <= 7) & (robust_zscores >= -3.5) & (robust_zscores <= 3.5) & (unmatch_dcat['ELLIPTICITY'] < 0.8)]
print(f'No. of likely astrophysical sources (flags and fwhm z-score and ellipticity): {len(unmatch_dcat_astrophysical)}/{len(unmatch_dcat)} ({100*len(unmatch_dcat_astrophysical)/len(unmatch_dcat)}%)')
print(f'No. of likely astrophysical sources (flags and fwhm z-score and ellipticity) brighter than refs limit mag: {len(unmatch_dcat_astrophysical[unmatch_dcat_astrophysical["MAG_ISO"] < maglim_ref])}')

# unmatch_dcat_astrophysical_cut = unmatch_dcat_astrophysical[dcut_condition_CHECK(unmatch_dcat_astrophysical)]
# print(f'No. of likely astrophysical sources satisfying selection criteria: {len(unmatch_dcat_astrophysical_cut)}')

num_magcut_astrophysical = (unmatch_dcat_astrophysical["MAG_ISO"] > maglim_ref) & (unmatch_dcat_astrophysical["MAG_ISO"] <= 24)
print(f'No. of likely astrophysical sources with 24 >= m > maglim: {np.sum(num_magcut_astrophysical)}/{len(unmatch_dcat_astrophysical)} ({100*np.mean(num_magcut_astrophysical)}%)')

# num_ellcut_astrophysical = (unmatch_dcat_astrophysical["ELLIPTICITY"] >= 0.6)
# print(f'No. of likely astrophysical sources with ellipticity >= 0.6: {np.sum(num_ellcut_astrophysical)}/{len(unmatch_dcat_astrophysical)} ({100*np.mean(num_ellcut_astrophysical)}%)')

print(f'Median mag, FWHM, and ellipticity of unmatched deconv = {np.round(np.median(unmatch_dcat_astrophysical["MAG_ISO"]), 2), np.round(np.median(unmatch_dcat_astrophysical["FWHM_IMAGE"]), 2), np.round(np.median(unmatch_dcat_astrophysical["ELLIPTICITY"]), 2)}')
print(f'Scatter about median of mag, FWHM and ellipticity of unmatched deconv = {np.round(median_abs_deviation(unmatch_dcat_astrophysical["MAG_ISO"]), 2), np.round(median_abs_deviation(unmatch_dcat_astrophysical["FWHM_IMAGE"]), 2), np.round(median_abs_deviation(unmatch_dcat_astrophysical["ELLIPTICITY"]), 2)}')

In [ ]:
print(ID)
one_to_one_cut.to_csv(f'one_to_one_cut_{ID}r.csv')

In [ ]:
unmatch_dcat_astrophysical[unmatch_dcat_astrophysical['MAG_ISO'] > maglim_sci].to_csv(f'unmatch_dcat_astrophysical_faint_{ID}r.csv')
unmatch_dcat_astrophysical_faint = pd.read_csv(f'/content/unmatch_dcat_astrophysical_faint_{ID}r.csv')
print(unmatch_dcat_astrophysical_faint.shape)
unmatch_dcat_astrophysical_faint['MAG_ISO'].min(), unmatch_dcat_astrophysical['MAG_ISO'].max(), unmatch_dcat_astrophysical_faint['FLAGS'].max()

In [ ]:
# added after 2nd review
for row in unmatch_dcat_astrophysical.iterrows():
    assert np.any(np.isclose(dcat['X_WORLD'] - row[1]['X_WORLD'], 0))
    assert np.any(np.isclose(dcat['Y_WORLD'] - row[1]['Y_WORLD'], 0))

In [ ]:
# added after 2nd review
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

robust_zscores_dcat_cut = (dcat_cut['FWHM_IMAGE'] - np.median(one_to_one_cut["FWHM_IMAGE_1"])) / (1.4826 * median_abs_deviation(one_to_one_cut["FWHM_IMAGE_1"]))
dcat_cut_astrophysical = dcat_cut[(dcat_cut['FLAGS'] <= 7) & (robust_zscores_dcat_cut >= -3.5) & (robust_zscores_dcat_cut <= 3.5) & (dcat_cut['ELLIPTICITY'] < 0.8)]

# dcat_cut[(dcat_cut['FLAGS'] <= 7) & (dcat_cut['MAG_ISO'] > maglim_sci) & (dcat_cut['MAG_ISO'] <= 24)]
# from scipy.spatial import KDTree, cKDTree
# tree = KDTree(dcat_cut_astrophysical[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])

unmatch_dcat_astrophysical_faint['closest_deconv_dist'] = [np.nan] * len(unmatch_dcat_astrophysical_faint)
unmatch_dcat_astrophysical_faint['whether_sextractor_deblended'] = [np.nan] * len(unmatch_dcat_astrophysical_faint)

deconvolved_catalog_table = Table.from_pandas(dcat_cut_astrophysical)

for row in unmatch_dcat_astrophysical_faint.iterrows():
    c = SkyCoord(ra=row[1]['X_WORLD']*u.degree, dec=row[1]['Y_WORLD']*u.degree)
    catalog = SkyCoord(ra=deconvolved_catalog_table['X_WORLD']*u.degree, dec=deconvolved_catalog_table['Y_WORLD']*u.degree)
    idx, d2d, d3d = c.match_to_catalog_sky(catalog, nthneighbor=2) # 2 because first will the point itself, for closest we want the second

    # ds, inds =  tree.query(
    #     (row[1]['X_IMAGE_DBL'], row[1]['Y_IMAGE_DBL']), 2  # 2 because first will the point itself, for closest we want the second
    # ) # finds the nearest neighbor
    unmatch_dcat_astrophysical_faint.at[row[0], 'closest_deconv_dist'] = (d2d.to(u.arcsec)).value[0]
    unmatch_dcat_astrophysical_faint.at[row[0], 'whether_sextractor_deblended'] = 1 if check_particular_flag(
        unmatch_dcat_astrophysical_faint.iloc[row[0]]['FLAGS'], nth_bit_from_right=1
    ) else 0

unmatch_dcat_astrophysical_faint[['NUMBER', 'ID_PARENT', 'X_WORLD', 'Y_WORLD', 'closest_deconv_dist', 'whether_sextractor_deblended', 'SUBDIV_NUMBER']].to_csv(f'unmatch_dcat_astrophysical_faint_{ID}r_condensed_moreinfo.csv')

In [ ]:
xx = unmatch_dcat_astrophysical[(unmatch_dcat_astrophysical['MAG_ISO'] > maglim_sci) & (unmatch_dcat_astrophysical['MAG_ISO'] <= 21.5)]
for row in xx.iterrows():
    assert np.any(np.isclose(row[1]['X_IMAGE_DBL'], dcat_cut['X_IMAGE_DBL']))
    assert np.any(np.isclose(row[1]['Y_IMAGE_DBL'], dcat_cut['Y_IMAGE_DBL']))

unmatch_dcat_astrophysical[(unmatch_dcat_astrophysical['MAG_ISO'] > maglim_sci) & (unmatch_dcat_astrophysical['MAG_ISO'] <= 21.5)].shape

In [ ]:
unmatch_dcat_astrophysical[(unmatch_dcat_astrophysical['MAG_ISO'] > 21.5) & (unmatch_dcat_astrophysical['MAG_ISO'] <= 24)].shape

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
# sns.set_style('ticks')
# sns.set_context('talk')  # change to 'paper' later.

import scienceplots
plt.style.use('science')

gen_gt_color = '#FC9272'
ip_gt_color = '#1C9099'
ip_gen_color = '#2ca25f'

ticklabelsize = 26
axeslabelsize = 27
titlesize = 30

linewidth = 4

In [ ]:
one_to_one_subset = one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > maglim_sci) & (one_to_one_cut['MAG_ISO_1'] <= 21.5)]

band_prefix = r'-$r$'
extent1 = (15, 22, 0, 4)
gridsize1 = (23, 19)
fig, ax = plt.subplots(1, 1, figsize=(8, 7))#, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
mappable = ax.hexbin(one_to_one_subset['MAG_ISO_1'], one_to_one_subset['FWHM_IMAGE_1'], C=one_to_one_subset['ELLIPTICITY_1'], mincnt=1, edgecolor='grey', cmap='OrRd_r', gridsize=gridsize1, extent=extent1, vmin=0, vmax=1)#, reduce_C_function=reduce_C_function)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar = plt.colorbar(mappable, ax=ax, orientation='vertical', pad=0.02, fraction=0.1)
cbar.ax.tick_params(labelsize=ticklabelsize-2)
cbar.set_label('ellipticity', fontsize=ticklabelsize, labelpad=10)
ax.set_title(title, fontsize=titlesize)

ax.axvline(x=np.median(one_to_one_subset['MAG_ISO_1']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
ax.axhline(y=np.median(one_to_one_subset['FWHM_IMAGE_1']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_ref, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_sci, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=21.5, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.fill_betweenx([0, 4], x1=maglim_sci, x2=21.5, alpha=0.15)

# ax.text(.05, .95, f'{len(unmatch_dcat_astrophysical_subset)} sources\nmedian ellipticity = {np.round(np.median(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3)
ax.text(.05, .95, f'{len(one_to_one_subset)} sources\nmag = {np.round(np.median(one_to_one_subset["MAG_ISO_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["MAG_ISO_1"]), 2)}\nFWHM = {np.round(np.median(one_to_one_subset["FWHM_IMAGE_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["FWHM_IMAGE_1"]), 2)} pix\nellipticity = {np.round(np.median(one_to_one_subset["ELLIPTICITY_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["ELLIPTICITY_1"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3, linespacing=1.5)

ax.set_xlabel(r'$m_{\mathrm{d}}$', fontsize=axeslabelsize)
ax.set_ylabel(r'$\mathrm{FWHM}_{\mathrm{d}}$ (pix)', fontsize=axeslabelsize)

ax.tick_params(which='major', axis='x', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='x', labelsize=ticklabelsize, length=3)
ax.tick_params(which='major', axis='y', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='y', labelsize=ticklabelsize, length=3)

ax.set_xticks([15, 16, 17, 18, 19, 20, 21, 22])
ax.set_yticks([0, 1, 2, 3, 4])

plt.savefig(f'mag_fwhm_ellipticity_one_to_one_deepref_experiment_{title.split("-")[0]}_r.pdf', bbox_inches='tight', dpi=200, format='pdf')
plt.clf()

plt.show()

In [ ]:
one_to_one_subset = one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > 21.5) & (one_to_one_cut['MAG_ISO_1'] <= 24)]

band_prefix = r'-$r$'
extent1 = (15, 24, 0, 4)
gridsize1 = (23, 19)
fig, ax = plt.subplots(1, 1, figsize=(8, 7))#, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
mappable = ax.hexbin(one_to_one_subset['MAG_ISO_1'], one_to_one_subset['FWHM_IMAGE_1'], C=one_to_one_subset['ELLIPTICITY_1'], mincnt=1, edgecolor='grey', cmap='OrRd_r', gridsize=gridsize1, extent=extent1, vmin=0, vmax=1)#, reduce_C_function=reduce_C_function)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar = plt.colorbar(mappable, ax=ax, orientation='vertical', pad=0.02, fraction=0.1)
cbar.ax.tick_params(labelsize=ticklabelsize-2)
cbar.set_label('ellipticity', fontsize=ticklabelsize, labelpad=10)
ax.set_title(title, fontsize=titlesize)

ax.axvline(x=np.median(one_to_one_subset['MAG_ISO_1']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
ax.axhline(y=np.median(one_to_one_subset['FWHM_IMAGE_1']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_ref, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_sci, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=21.5, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.fill_betweenx([0, 4], x1=maglim_sci, x2=21.5, alpha=0.15)

# ax.text(.05, .95, f'{len(unmatch_dcat_astrophysical_subset)} sources\nmedian ellipticity = {np.round(np.median(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3)
ax.text(.05, .95, f'{len(one_to_one_subset)} sources\nmag = {np.round(np.median(one_to_one_subset["MAG_ISO_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["MAG_ISO_1"]), 2)}\nFWHM = {np.round(np.median(one_to_one_subset["FWHM_IMAGE_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["FWHM_IMAGE_1"]), 2)} pix\nellipticity = {np.round(np.median(one_to_one_subset["ELLIPTICITY_1"]), 2)} $\pm$ {np.round(median_abs_deviation(one_to_one_subset["ELLIPTICITY_1"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3, linespacing=1.5)

ax.set_xlabel(r'$m_{\mathrm{d}}$', fontsize=axeslabelsize)
ax.set_ylabel(r'$\mathrm{FWHM}_{\mathrm{d}}$ (pix)', fontsize=axeslabelsize)

ax.tick_params(which='major', axis='x', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='x', labelsize=ticklabelsize, length=3)
ax.tick_params(which='major', axis='y', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='y', labelsize=ticklabelsize, length=3)

ax.set_xticks([15, 16, 17, 18, 19, 20, 21, 22, 23, 24])
ax.set_yticks([0, 1, 2, 3, 4])

plt.savefig(f'mag_fwhm_ellipticity_one_to_one_faint_deepref_experiment_{title.split("-")[0]}_r.pdf', bbox_inches='tight', dpi=200, format='pdf')
plt.clf()

plt.show()

In [ ]:
unmatch_dcat_astrophysical_subset = unmatch_dcat_astrophysical[(unmatch_dcat_astrophysical['MAG_ISO'] > maglim_sci) & (unmatch_dcat_astrophysical['MAG_ISO'] <= 21.5)]

band_prefix = r'-$r$'
extent1 = (15, 22, 0, 4)
gridsize1 = (23, 19)
fig, ax = plt.subplots(1, 1, figsize=(8, 7))#, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
mappable = ax.hexbin(unmatch_dcat_astrophysical_subset['MAG_ISO'], unmatch_dcat_astrophysical_subset['FWHM_IMAGE'], C=unmatch_dcat_astrophysical_subset['ELLIPTICITY'], mincnt=1, edgecolor='grey', cmap='OrRd_r', gridsize=gridsize1, extent=extent1, vmin=0, vmax=1)#, reduce_C_function=reduce_C_function)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar = plt.colorbar(mappable, ax=ax, orientation='vertical', pad=0.02, fraction=0.1)
cbar.ax.tick_params(labelsize=ticklabelsize-2)
cbar.set_label('ellipticity', fontsize=ticklabelsize, labelpad=10)
ax.set_title(title, fontsize=titlesize)

ax.axvline(x=np.median(unmatch_dcat_astrophysical_subset['MAG_ISO']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
ax.axhline(y=np.median(unmatch_dcat_astrophysical_subset['FWHM_IMAGE']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_ref, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=maglim_sci, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.axvline(x=21.5, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
# ax.fill_betweenx([0, 4], x1=maglim_sci, x2=21.5, alpha=0.15)

# ax.text(.05, .95, f'{len(unmatch_dcat_astrophysical_subset)} sources\nmedian ellipticity = {np.round(np.median(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3)
ax.text(.05, .95, f'{len(unmatch_dcat_astrophysical_subset)} sources\nmag = {np.round(np.median(unmatch_dcat_astrophysical_subset["MAG_ISO"]), 2)} $\pm$ {np.round(median_abs_deviation(unmatch_dcat_astrophysical_subset["MAG_ISO"]), 2)}\nFWHM = {np.round(np.median(unmatch_dcat_astrophysical_subset["FWHM_IMAGE"]), 2)} $\pm$ {np.round(median_abs_deviation(unmatch_dcat_astrophysical_subset["FWHM_IMAGE"]), 2)} pix\nellipticity = {np.round(np.median(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)} $\pm$ {np.round(median_abs_deviation(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3, linespacing=1.5)

ax.set_xlabel(r'$m_{\mathrm{d}}$', fontsize=axeslabelsize)
ax.set_ylabel(r'$\mathrm{FWHM}_{\mathrm{d}}$ (pix)', fontsize=axeslabelsize)

ax.tick_params(which='major', axis='x', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='x', labelsize=ticklabelsize, length=3)
ax.tick_params(which='major', axis='y', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='y', labelsize=ticklabelsize, length=3)

ax.set_xticks([15, 16, 17, 18, 19, 20, 21, 22])
ax.set_yticks([0, 1, 2, 3, 4])

plt.savefig(f'mag_fwhm_ellipticity_unmatched_deconvolved_deepref_experiment_{title.split("-")[0]}_r.pdf', bbox_inches='tight', dpi=200, format='pdf')
plt.clf()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
# sns.set_style('ticks')
# sns.set_context('talk')  # change to 'paper' later.

# import scienceplots
# plt.style.use('science')

gen_gt_color = '#FC9272'
ip_gt_color = '#1C9099'
ip_gen_color = '#2ca25f'

ticklabelsize = 26
axeslabelsize = 27
titlesize = 30

linewidth = 4

band_prefix = r'-$r$'
extent1 = (15, 24, 0, 4)
gridsize1 = (19, 15)
fig, ax = plt.subplots(1, 1, figsize=(8, 7))#, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
mappable = ax.hexbin(unmatch_dcat_astrophysical_subset['MAG_ISO'], unmatch_dcat_astrophysical_subset['FWHM_IMAGE'], mincnt=1, edgecolor='grey', cmap='OrRd', gridsize=gridsize1, extent=extent1)#, reduce_C_function=reduce_C_function)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar = plt.colorbar(mappable, ax=ax, orientation='vertical', pad=0.02, fraction=0.1)
cbar.ax.tick_params(labelsize=ticklabelsize-2)
cbar.set_label('Counts', fontsize=ticklabelsize, labelpad=10)
ax.set_title(title, fontsize=titlesize)

ax.axvline(x=np.median(unmatch_dcat_astrophysical_subset['MAG_ISO']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
ax.axhline(y=np.median(unmatch_dcat_astrophysical_subset['FWHM_IMAGE']), linestyle='--', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)
ax.axvline(x=maglim_ref, linestyle='dotted', linewidth=linewidth-2, color='#0a0a0a', alpha=0.85)

ax.text(.05, .95, f'{len(unmatch_dcat_astrophysical_subset)} sources\nmedian ellipticity = {np.round(np.median(unmatch_dcat_astrophysical_subset["ELLIPTICITY"]), 2)}', ha='left', va='top', transform=ax.transAxes, fontsize=axeslabelsize-3)

ax.set_xlabel(r'$m_{\mathrm{d}}$', fontsize=axeslabelsize)
ax.set_ylabel(r'$\mathrm{FWHM}_{\mathrm{d}}$ (pix)', fontsize=axeslabelsize)

ax.tick_params(which='major', axis='x', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='x', labelsize=ticklabelsize, length=3)
ax.tick_params(which='major', axis='y', labelsize=ticklabelsize, length=6)
ax.tick_params(which='minor', axis='y', labelsize=ticklabelsize, length=3)

ax.set_xticks([15, 16, 17, 18, 19, 20, 21, 22, 23, 24])
ax.set_yticks([0, 1, 2, 3, 4])

plt.show()

In [ ]:
unmatch_dcat_astrophysical['FLAGS'].value_counts()

In [ ]:
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

ref_wcs = WCS(fits.open(f'/content/{CROSSMATCH_PREFIX}')[0].header)

count = 0
for row in unmatch_dcat_astrophysical.iterrows():
    contains_or_not = ref_wcs.footprint_contains(
        SkyCoord(
            ra=row[1]['X_WORLD'],
            dec=row[1]['Y_WORLD'],
            unit='deg'
        )
    )
    if not contains_or_not:
        count += 1

if count == 0:
    print('There is no source in the unmatched deconv catalog that does not lie in the sky footprint of the reference image.')
else:
    print(f'There are {count} sources in the unmatched deconv catalog that does not lie in the sky footprint of the reference image.')

In [ ]:
#########################################################################################################
DECONV_FILEPATH = '/content/ztf_deconv_results/sgp_reconstruction_results/deconvolved_subdiv_ztf_20221120493183_000626_zr_c12_o_q2_sciimg.fits'
deconv_img = fits.getdata(DECONV_FILEPATH)
#########################################################################################################

In [ ]:
from astropy.wcs.utils import skycoord_to_pixel
from astropy.nddata import NoOverlapError
import astropy.units as u
from astropy.nddata import Cutout2D
from matplotlib.patches import Ellipse

w1 = WCS(fits.open(CROSSMATCH_PREFIX)[0].header)
w2 = WCS(fits.open(DECONV_FILEPATH)[0].header)

unmatch_dcat_astrophysical_shuffled = unmatch_dcat_astrophysical.sample(frac=1, random_state=42)

for example in unmatch_dcat_astrophysical_shuffled.iterrows():
    _oc = (example[1]['X_WORLD'], example[1]['Y_WORLD'])
    _ocoord = SkyCoord(ra=_oc[0]*u.deg, dec=_oc[1]*u.deg)
    _ocoord_pixel = skycoord_to_pixel(_ocoord, wcs=w1)
    # _ocoord_pixel_for_deconv = skycoord_to_pixel(_ocoord, wcs=w2)

    # print(_ocoord_pixel, _ocoord_pixel_for_deconv)

    # NOTE: THIS BELOW CONDITION IS RELIABLE ONLY FOR UNCROWDED FIELDS, OTHERWISE DUBIOUS MATCHES MAY BE PRESENT.
    # NOTE: ALSO NOTE THAT WE USE dcat INSTEAD dcat_cut.
    # is_deconv_source_present = dcat_cut[(dcat_cut['X_IMAGE_DBL'] >= _ocoord[0]-2) & (dcat_cut['X_IMAGE_DBL'] <= _ocoord[0]+2) & (dcat_cut['Y_IMAGE_DBL'] >= _ocoord[1]-2) & (dcat_cut['Y_IMAGE_DBL'] <= _ocoord[1]+2)]
    rect_d = 7
    is_orig_source_present = ocat[(ocat['X_WORLD'] >= _oc[0]-(rect_d/3600)) & (ocat['X_WORLD'] <= _oc[0]+(rect_d/3600)) & (ocat['Y_WORLD'] >= _oc[1]-(rect_d/3600)) & (ocat['Y_WORLD'] <= _oc[1]+(rect_d/3600))]
    # is_deconv_source_present2 = dcat_cut[]
    if len(is_orig_source_present) > 0:
        print(f'orig source was present using a higher position crossmatch threshold!! and has mag = {is_orig_source_present["MAG_ISO"].iloc[0]}')

    ocut = Cutout2D(deconv_img, _ocoord, size=50, wcs=w2)

    try:
        dcut = Cutout2D(fits.getdata(CROSSMATCH_PREFIX), _ocoord, size=50, wcs=w1)
    except NoOverlapError:
        continue

    # print(_ocoord)
    # print(ocut.data.min(), ocut.data.max())
    # print(dcut.data.min(), dcut.data.max())

    _oocoord = ocut.to_cutout_position(_ocoord_pixel)
    # _ddcoord = dcut.to_cutout_position(_ocoord)

    vmin = np.min([ocut.data.min(), dcut.data.min()])
    vmax = np.max([ocut.data.max(), dcut.data.max()])

    if np.all(np.isnan(dcut.data)):
        continue
    dnorm = ImageNormalize(dcut.data, interval=PercentileInterval(99.5),
                    stretch=SqrtStretch())#, vmin=vmin, vmax=vmax)
    onorm = ImageNormalize(ocut.data, interval=PercentileInterval(99.5),
                    stretch=SqrtStretch())#, vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
    ax[0].imshow(ocut.data, norm=onorm, interpolation='nearest', origin='lower', cmap='inferno')
    # ax[0].set_title('Original', fontsize=titlesize)
    # ax[0].set_title('Deconvolved', fontsize=titlesize)
    # ax[0].set_title(f'Orig: MAG_ISO={str(np.round(example["MAG_ISO_2"].iloc[0], 2))}\n(X, Y)={np.round(_ocoord, 2)}')
    # ax[1].set_title(f'Deconv: {np.round(lis(t(example["MAG_ISO_1"]), 2)}\n{tuple(np.round(_dcoord1, 2)), tuple(np.round(_dcoord2, 2))}')
    # fig.suptitle(f'col1_2 = {col}')
    # after first review: added hflip as a hacky way. Because the diff between _align1 and normal is that it's hflipped. so doing this is the correct way assuming this.
    im = ax[1].imshow(np.fliplr(dcut.data), norm=dnorm, interpolation='nearest', origin='lower', cmap='inferno')
    # cax = ax[1].inset_axes([1.05, 0.0, 0.05, 1.])
    # cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    # cbar.ax.tick_params(labelsize=ticklabelsize-2)

    # ax[0].scatter(_oocoord[0], _oocoord[1], c='red', marker='4')
    # ax[1].scatter(_oocoord[0], _oocoord[1], c='red', marker='4')

    mul_factor = 6
    e = Ellipse(xy=_oocoord,
        width=mul_factor*example[1]['A_IMAGE'],
        height=mul_factor*example[1]['B_IMAGE'],
        angle=example[1]['THETA_IMAGE'] * 180. / np.pi,
        linewidth=1.1
    )
    e.set_facecolor('none')
    e.set_edgecolor('#a8ddb5')
    ax[0].add_artist(e)

    ax[0].set_xticks([]); ax[0].set_yticks([])
    ax[1].set_xticks([]); ax[1].set_yticks([])

    text_x, text_y = .1, .95
    ax[0].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(example[1]["FWHM_IMAGE"], 2)}, {np.round(example[1]["ELLIPTICITY"], 2)}, {np.round(example[1]["MAG_ISO"], 2)}, {int(example[1]["FLAGS"])}', ha='left', va='top', transform=ax[0].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)

    if len(is_orig_source_present) > 0:
        mul_factor = 6
        for example in is_orig_source_present.iterrows():
            _dddcoord = dcut.to_cutout_position((example[1]['X_WORLD'], example[1]['Y_WORLD']))
            e = Ellipse(xy=_dddcoord,
                width=mul_factor*example[1]['A_IMAGE'],
                height=mul_factor*example[1]['B_IMAGE'],
                angle=example[1]['THETA_IMAGE'] * 180. / np.pi,
                linewidth=1.1
            )
            e.set_facecolor('none')
            e.set_edgecolor('#a8ddb5')
            ax[1].add_artist(e)
            ax[1].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(example[1]["FWHM_IMAGE"], 2)}, {np.round(example[1]["ELLIPTICITY"], 2)}, {np.round(example[1]["MAG_ISO"], 2)}, {int(example[1]["FLAGS"])}', ha='left', va='top', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)
            break

    plt.show()

In [ ]:
plt.scatter(unmatch_dcat_astrophysical['X_IMAGE_DBL'], unmatch_dcat_astrophysical['Y_IMAGE_DBL'])

In [ ]:
######################################################################################################################################
ORIG_FILEPATH = '/content/ztf_deconv_results/sgp_reconstruction_results/orig_subdiv_ztf_20221120493183_000626_zr_c12_o_q2_sciimg.fits'
######################################################################################################################################

In [ ]:
from astropy.wcs.utils import skycoord_to_pixel
from astropy.nddata import NoOverlapError
import astropy.units as u

w1 = WCS(fits.open(CROSSMATCH_PREFIX)[0].header)
w2 = WCS(fits.open(DECONV_FILEPATH)[0].header)
w2 = WCS(fits.open(ORIG_FILEPATH)[0].header)
# deconv_img = fits.getdata(ORIG_FILEPATH)

unmatch_ocat = pd.read_csv(os.path.join(directory, 'unmatches_original_cut_' + prefix.split('/')[-1] + '.csv'))
unmatch_ocat_shuffled = unmatch_ocat.sample(frac=1, random_state=42)

unmatch_ocat_shuffled_case1 = unmatch_ocat_shuffled[(unmatch_ocat_shuffled['MAG_ISO'] > maglim_sci) & (unmatch_ocat_shuffled['MAG_ISO'] <= 21.5)]

for example in unmatch_ocat_shuffled_case1.iterrows():
    _oc = (example[1]['X_WORLD'], example[1]['Y_WORLD'])
    _ocoord = SkyCoord(ra=_oc[0]*u.deg, dec=_oc[1]*u.deg)
    _ocoord_pixel = skycoord_to_pixel(_ocoord, wcs=w1)
    # _ocoord_pixel_for_deconv = skycoord_to_pixel(_ocoord, wcs=w2)

    # print(_ocoord_pixel, _ocoord_pixel_for_deconv)

    # NOTE: THIS BELOW CONDITION IS RELIABLE ONLY FOR UNCROWDED FIELDS, OTHERWISE DUBIOUS MATCHES MAY BE PRESENT.
    # NOTE: ALSO NOTE THAT WE USE dcat INSTEAD dcat_cut.
    # is_deconv_source_present = dcat_cut[(dcat_cut['X_IMAGE_DBL'] >= _ocoord[0]-2) & (dcat_cut['X_IMAGE_DBL'] <= _ocoord[0]+2) & (dcat_cut['Y_IMAGE_DBL'] >= _ocoord[1]-2) & (dcat_cut['Y_IMAGE_DBL'] <= _ocoord[1]+2)]
    rect_d = 7
    is_deconv_source_present = dcat[(dcat['X_WORLD'] >= _oc[0]-(rect_d/3600)) & (dcat['X_WORLD'] <= _oc[0]+(rect_d/3600)) & (dcat['Y_WORLD'] >= _oc[1]-(rect_d/3600)) & (dcat['Y_WORLD'] <= _oc[1]+(rect_d/3600))]
    # is_deconv_source_present2 = dcat_cut[]
    if len(is_deconv_source_present) > 0:
        print(f'deconv source was present using a higher position crossmatch threshold!! and has mag = {is_deconv_source_present["MAG_ISO"].iloc[0]}')

    ocut = Cutout2D(fits.getdata(CROSSMATCH_PREFIX), _ocoord, size=50, wcs=w1)

    try:
        dcut = Cutout2D(deconv_img, _ocoord, size=50, wcs=w2)
    except NoOverlapError:
        continue

    # print(_ocoord)
    # print(ocut.data.min(), ocut.data.max())
    # print(dcut.data.min(), dcut.data.max())

    _oocoord = ocut.to_cutout_position(_ocoord_pixel)
    # _ddcoord = dcut.to_cutout_position(_ocoord)

    vmin = np.min([ocut.data.min(), dcut.data.min()])
    vmax = np.max([ocut.data.max(), dcut.data.max()])

    if np.all(np.isnan(dcut.data)):
        continue
    dnorm = ImageNormalize(dcut.data, interval=PercentileInterval(99.5),
                    stretch=SqrtStretch())#, vmin=vmin, vmax=vmax)
    onorm = ImageNormalize(ocut.data, interval=PercentileInterval(99.5),
                    stretch=SqrtStretch())#, vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
    ax[0].imshow(ocut.data, norm=onorm, interpolation='nearest', origin='lower', cmap='inferno')
    # ax[0].set_title('Original', fontsize=titlesize)
    # ax[0].set_title('Deconvolved', fontsize=titlesize)
    # ax[0].set_title(f'Orig: MAG_ISO={str(np.round(example["MAG_ISO_2"].iloc[0], 2))}\n(X, Y)={np.round(_ocoord, 2)}')
    # ax[1].set_title(f'Deconv: {np.round(lis(t(example["MAG_ISO_1"]), 2)}\n{tuple(np.round(_dcoord1, 2)), tuple(np.round(_dcoord2, 2))}')
    # fig.suptitle(f'col1_2 = {col}')
    # after first review: added hflip as a hacky way. Because the diff between _align1 and normal is that it's hflipped. so doing this is the correct way assuming this.
    im = ax[1].imshow(np.fliplr(dcut.data), norm=dnorm, interpolation='nearest', origin='lower', cmap='inferno')
    # cax = ax[1].inset_axes([1.05, 0.0, 0.05, 1.])
    # cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    # cbar.ax.tick_params(labelsize=ticklabelsize-2)

    # ax[0].scatter(_oocoord[0], _oocoord[1], c='red', marker='4')
    # ax[1].scatter(_oocoord[0], _oocoord[1], c='red', marker='4')

    mul_factor = 6
    e = Ellipse(xy=_oocoord,
        width=mul_factor*example[1]['A_IMAGE'],
        height=mul_factor*example[1]['B_IMAGE'],
        angle=example[1]['THETA_IMAGE'] * 180. / np.pi,
        linewidth=1.1
    )
    e.set_facecolor('none')
    e.set_edgecolor('#a8ddb5')
    ax[0].add_artist(e)

    ax[0].set_xticks([]); ax[0].set_yticks([])
    ax[1].set_xticks([]); ax[1].set_yticks([])

    text_x, text_y = .1, .95
    ax[0].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(example[1]["FWHM_IMAGE"], 2)}, {np.round(example[1]["ELLIPTICITY"], 2)}, {np.round(example[1]["MAG_ISO"], 2)}, {int(example[1]["FLAGS"])}', ha='left', va='top', transform=ax[0].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)

    if len(is_deconv_source_present) > 0:
        mul_factor = 6
        for example in is_orig_source_present.iterrows():
            _dddcoord = dcut.to_cutout_position((example[1]['X_WORLD'], example[1]['Y_WORLD']))
            e = Ellipse(xy=_dddcoord,
                width=mul_factor*example[1]['A_IMAGE'],
                height=mul_factor*example[1]['B_IMAGE'],
                angle=example[1]['THETA_IMAGE'] * 180. / np.pi,
                linewidth=1.1
            )
            e.set_facecolor('none')
            e.set_edgecolor('#a8ddb5')
            ax[1].add_artist(e)
            ax[1].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(example[1]["FWHM_IMAGE"], 2)}, {np.round(example[1]["ELLIPTICITY"], 2)}, {np.round(example[1]["MAG_ISO"], 2)}, {int(example[1]["FLAGS"])}', ha='left', va='top', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)
            break

    plt.show()

In [ ]:
unmatch_ocat['SUBDIV_NUMBER'].hist()

In [ ]:
plt.scatter(unmatch_ocat['X_IMAGE_DBL'], unmatch_ocat['Y_IMAGE_DBL'])

In [ ]:
################ Search for blended original sources ################
import math
import math
def blended_or_not(orow, drow, norm_factor=1.):
    """See Eq.1 of https://iopscience.iop.org/article/10.3847/0004-637X/816/1/11.

    Assumes many_to_one_subset contains only two rows.

    Returns True if ambiguously blended, else False.
    """
    # if np.isclose(many_to_one_subset['FWHM_IMAGE_1'].iloc[0], 0.0) or np.isclose(many_to_one_subset['FLAGS_1'].iloc[1], 0.0):
    # if many_to_one_subset['FWHM_IMAGE_1'].iloc[0] < 0.1 or many_to_one_subset['FWHM_IMAGE_1'].iloc[1] < 0.1:
    #     return False
    # # First check flags.
    # if many_to_one_subset['FLAGS_1'].iloc[0] > 7 or many_to_one_subset['FLAGS_1'].iloc[0] > 7:
    #     return False
    # ocoord = (many_to_one_subset['X_IMAGE_DBL_2'].iloc[0], many_to_one_subset['Y_IMAGE_DBL_2'].iloc[0])
    # dcoord1 = (many_to_one_subset['X_IMAGE_DBL_1'].iloc[0], many_to_one_subset['Y_IMAGE_DBL_1'].iloc[0])
    # assert len(orow) == 1
    # assert len(drow) == 1
    ocoord = tuple(orow[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
    dcoord1 = tuple(drow[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
    dist1 = math.dist(ocoord, dcoord1)
    sigma_j = orow['FWHM_IMAGE']
    if np.isclose(sigma_j, 0.0):
        return False
    # sigma_i1 = many_to_one_subset['FWHM_IMAGE_1'].iloc[0]
    # sigma_i2 = many_to_one_subset['FWHM_IMAGE_1'].iloc[1]
    # We approximate the Gaussian-convolved FWHM of the deconvolved with the original source's FWHM for simplicity.
    sigma_i1, sigma_i2 = sigma_j, sigma_j
    dist_eff1 = dist1 / (norm_factor * (sigma_j + sigma_i1))
    condition1 = dist_eff1 < 1
    # This condition is more stricter to ensure a purer deblended sample.
    condition2 = (
        orow['ELLIPTICITY'] > drow['ELLIPTICITY'] - 0.05
    )
    return condition1 and condition2

def find_closest_orig_to_deconv(orig_df, deconv_row):
    min_d = np.Inf
    for i, row in enumerate(orig_df.iterrows()):
        d = math.dist(
            tuple(row[1][['X_IMAGE_DBL', 'Y_IMAGE_DBL']]),
            tuple(deconv_row[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
        )
        if d < min_d:
            min_d = d
            min_row_number = i

    return orig_df.iloc[i]

def mag_to_flux(m):
    return 10 ** (0.4 * (magzp - m))

def flux_to_mag(flux):
    m = magzp - 2.5 * np.log10(flux) if flux > 0 else 99
    return m

def reduce_C_function(C):
    C = np.array(C)
    magzp = image_header['MAGZP']

    flux_sum = np.sum([mag_to_flux(c) for c in C])
    return flux_to_mag(flux_sum)

hdul = fits.open(CROSSMATCH_PREFIX)
image_header = hdul[0].header
magzp = image_header['MAGZP']
maglim = image_header["MAGLIM"]

# NOTE: ALSO NOTE THAT WE USE ocat INSTEAD ocat_cut.
from astropy.nddata import Cutout2D
from matplotlib.patches import Ellipse

orig_img = fits.getdata(IMG_NAMES[0])
# deconv_img = fits.getdata(IMG_NAMES[0].replace('orig_subdiv', 'deconvolved_subdiv'))
deconv_img = orig_img
rect_d = 5
count = 0
visualize_cutouts = True

deblend_full_data = []
most_closest_orig_sources = []
deconv_sources = []
already_done_orig_sources = []

for row in unmatch_dcat_astrophysical.iterrows():

    closest_orig_source = ocat[(ocat['X_IMAGE_DBL'] >= row[1]['X_IMAGE_DBL']-rect_d) & (ocat['X_IMAGE_DBL'] <= row[1]['X_IMAGE_DBL']+rect_d) & (ocat['Y_IMAGE_DBL'] >= row[1]['Y_IMAGE_DBL']-rect_d) & (ocat['Y_IMAGE_DBL'] <= row[1]['Y_IMAGE_DBL']+rect_d)]
    closest_deconv_source = dcat[(dcat['X_IMAGE_DBL'] >= row[1]['X_IMAGE_DBL']-rect_d) & (dcat['X_IMAGE_DBL'] <= row[1]['X_IMAGE_DBL']+rect_d) & (dcat['Y_IMAGE_DBL'] >= row[1]['Y_IMAGE_DBL']-rect_d) & (dcat['Y_IMAGE_DBL'] <= row[1]['Y_IMAGE_DBL']+rect_d)]
    closest_deconv_source = closest_deconv_source[(closest_deconv_source['X_IMAGE_DBL'] != row[1]['X_IMAGE_DBL']) & (closest_deconv_source['Y_IMAGE_DBL'] != row[1]['Y_IMAGE_DBL'])]

    # Only when this deconv source has an original counterpart and also another deconvolved source, it might be a deblend.
    if len(closest_orig_source) > 0 and len(closest_deconv_source) > 0:
        if len(closest_orig_source) > 1:
            most_closest_orig_source = find_closest_orig_to_deconv(closest_orig_source, row[1])
        elif len(closest_orig_source) == 1:
            closest_orig_source = closest_orig_source.iloc[0]
            most_closest_orig_source = closest_orig_source

        # We are only interested in orig converted into two deconv sources.
        # One is the one on which are looping through unmatch_dcat and the other is in `closest_deconv_source`.
        if len(closest_deconv_source) != 1:
            continue

        ################# THESE ARE LESS IMPORTANT CONDITIONS BUT STILL USEFUL FOR OBTAINING PURITY #################
        if closest_deconv_source['ELLIPTICITY'].iloc[0] >= 0.6 or row[1]['ELLIPTICITY'] >= 0.6:  # we only want deconvolved sources with ellipticity < 0.6.
            continue

        if np.isclose(closest_deconv_source['FWHM_IMAGE'].iloc[0], 0) or np.isclose(row[1]['FWHM_IMAGE'], 0):
            continue

        if most_closest_orig_source['ELLIPTICITY'] < 0.1:
            continue
        #############################################################################################################

        if most_closest_orig_source['FLAGS'] > 7:
            continue

        if check_particular_flag(most_closest_orig_source['FLAGS'], nth_bit_from_right=1):  # deblending flag for the orig must not be set.
            continue

        if most_closest_orig_source['NUMBER'] in already_done_orig_sources:
            continue

        # We use the most closest original source for deblending check.
        orig_blended_or_not = blended_or_not(most_closest_orig_source, row[1], norm_factor=1.)
        # We are only interested when the orig source is blended.
        if orig_blended_or_not:
            count += 1

            # after first review
            # print(most_closest_orig_source.shape)
            # print(closest_deconv_source.T.squeeze().shape)
            # print(row[1].shape)
            deblend_full_data.append((
                ID + f'_r',  # NOTE: Assuming we only select r-band data.
                most_closest_orig_source,
                closest_deconv_source.T.squeeze(),
                row[1]
            ))

            # if count == 40:
            #     break
            already_done_orig_sources.append(most_closest_orig_source['NUMBER'])
            dist_bet_deconv_sources = math.dist(
                (row[1]['X_IMAGE_DBL'], row[1]['Y_IMAGE_DBL']),
                (closest_deconv_source['X_IMAGE_DBL'].iloc[0], closest_deconv_source['Y_IMAGE_DBL'].iloc[0])
            )
            if dist_bet_deconv_sources > 2.5:
                continue
            if dist_bet_deconv_sources < 2.5:
                print('GOT A SOURCE WITH < 2.5 PIX SEPARATION!!!')
            most_closest_orig_sources.append(most_closest_orig_source)
            deconv_sources.append((
                [closest_deconv_source['ELLIPTICITY'].iloc[0], row[1]['ELLIPTICITY']],
                [closest_deconv_source['MAG_ISO'].iloc[0], row[1]['MAG_ISO']],
                dist_bet_deconv_sources
            ))

            ##### Comment from this line onwards if only statistics are needed and no visualization ######
            # print(count)
            if visualize_cutouts:
                ocoord = (most_closest_orig_source['X_IMAGE_DBL'], most_closest_orig_source['Y_IMAGE_DBL'])

                dcut = Cutout2D(deconv_img, ocoord, size=30)
                ocut = Cutout2D(orig_img, ocoord, size=30)
                onorm = ImageNormalize(ocut.data, interval=PercentileInterval(99.5),
                                stretch=SqrtStretch())
                dnorm = ImageNormalize(dcut.data, interval=PercentileInterval(99.5),
                                stretch=SqrtStretch())
                fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
                ax[0].imshow(ocut.data, origin='lower', interpolation='nearest', cmap='inferno', norm=onorm)
                ax[1].imshow(dcut.data, origin='lower', interpolation='nearest', cmap='inferno', norm=dnorm)

                text_x, text_y = .1, .95
                ax[0].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(most_closest_orig_source["FWHM_IMAGE"], 2)} pix, {np.round(most_closest_orig_source["ELLIPTICITY"], 2)}, {np.round(most_closest_orig_source["MAG_ISO"], 2)}, {int(most_closest_orig_source["FLAGS"])}', ha='left', va='top', transform=ax[0].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)
                ax[1].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(row[1]["FWHM_IMAGE"], 2)} pix, {np.round(row[1]["ELLIPTICITY"], 2)}, {np.round(row[1]["MAG_ISO"], 2)}, {int(row[1]["FLAGS"])}\n{np.round(closest_deconv_source["FWHM_IMAGE"].iloc[0], 2)} pix, {np.round(closest_deconv_source["ELLIPTICITY"].iloc[0], 2)}, {np.round(closest_deconv_source["MAG_ISO"].iloc[0], 2)}, {int(closest_deconv_source["FLAGS"].iloc[0])}', ha='left', va='top', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)
                ax[1].text(0.3, 0.3, f'dist = {dist_bet_deconv_sources:.2f} pix', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)
                total_mag = flux_to_mag(mag_to_flux(row[1]["MAG_ISO"]) + mag_to_flux(closest_deconv_source["MAG_ISO"].iloc[0]))
                ax[1].text(0.25, 0.2, f'total mag = {total_mag:.2f}', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize-10, weight='bold', family='sans-serif', linespacing=1.5)

                oo = ocut.to_cutout_position(ocoord)
                dd = dcut.to_cutout_position((row[1]['X_IMAGE_DBL'], row[1]['Y_IMAGE_DBL']))
                dd2 = dcut.to_cutout_position((closest_deconv_source['X_IMAGE_DBL'].iloc[0], closest_deconv_source['Y_IMAGE_DBL'].iloc[0]))

                mul_factor = 6
                e = Ellipse(xy=oo,
                    width=mul_factor*most_closest_orig_source['A_IMAGE'],
                    height=mul_factor*most_closest_orig_source['B_IMAGE'],
                    angle=most_closest_orig_source['THETA_IMAGE'] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[0].add_artist(e)

                mul_factor = 12
                e = Ellipse(xy=dd,
                    width=mul_factor*row[1]['A_IMAGE'],
                    height=mul_factor*row[1]['B_IMAGE'],
                    angle=row[1]['THETA_IMAGE'] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[1].add_artist(e)

                mul_factor = 12
                e = Ellipse(xy=dd2,
                    width=mul_factor*closest_deconv_source['A_IMAGE'].iloc[0],
                    height=mul_factor*closest_deconv_source['B_IMAGE'].iloc[0],
                    angle=closest_deconv_source['THETA_IMAGE'].iloc[0] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[1].add_artist(e)

                for a in ax:
                    a.set_xticks([])
                    a.set_yticks([])

                # plt.savefig(f'deblend_example_{title}_{count}.pdf', bbox_inches='tight', dpi=200, format='pdf')
                # plt.clf()

                plt.show()

print(f'{count} blended sources found')

---OLD---

In [ ]:
dcat_cut_astrophysical_faint = dcat_cut[(dcat_cut['FLAGS'] <= 7) & (dcat_cut['MAG_ISO'] > maglim_sci) & (dcat_cut['MAG_ISO'] <= 24)]
one_to_one_cut_astrophysical_faint = one_to_one_cut[(one_to_one_cut['MAG_ISO_1'] > maglim_sci) & (one_to_one_cut['MAG_ISO_1'] <= 24) & (one_to_one_cut['FLAGS_1'] <= 7)]
unmatches_deconvolved_cut_astrophysical_faint = unmatches_deconvolved_cut[(unmatches_deconvolved_cut['FLAGS'] <= 7) & (unmatches_deconvolved_cut['MAG_ISO'] > maglim_sci) & (unmatches_deconvolved_cut['MAG_ISO'] <= 24)]

dcat_cut.shape, dcat_cut_astrophysical_faint.shape, one_to_one_cut.shape, one_to_one_cut_astrophysical_faint.shape, unmatches_deconvolved_cut.shape, unmatches_deconvolved_cut_astrophysical_faint.shape

In [ ]:
# after first review.
unmatches_deconvolved_cut_astrophysical = unmatches_deconvolved_cut[unmatches_deconvolved_cut['FLAGS'] <= 7]
# see https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=terms-modified-z-score
# 1.22 is the median of deconv sources from the 1-1 match for 626-r and 0.25 is the MAD.
robust_zscores = (unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'] - 1.22) / (1.486 * 0.25)
robust_zscores_only = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3)]
print(f'No. of unreliable deconv sources using zscore criteria = {len(robust_zscores_only)}/{len(unmatches_deconvolved_cut_astrophysical)}')
robust_zscores_fwhmellzero = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.)))]
print(f'No. of unreliable deconv sources using zscore criteria and fwhm/ell near zero criteria = {len(robust_zscores_fwhmellzero)}/{len(unmatches_deconvolved_cut_astrophysical)}')

# after first review END.

In [ ]:
!pip install regions

In [ ]:
import math
def blended_or_not(orow, drow, norm_factor=1.):
    """See Eq.1 of https://iopscience.iop.org/article/10.3847/0004-637X/816/1/11.

    Assumes many_to_one_subset contains only two rows.

    Returns True if ambiguously blended, else False.
    """
    # if np.isclose(many_to_one_subset['FWHM_IMAGE_1'].iloc[0], 0.0) or np.isclose(many_to_one_subset['FLAGS_1'].iloc[1], 0.0):
    # if many_to_one_subset['FWHM_IMAGE_1'].iloc[0] < 0.1 or many_to_one_subset['FWHM_IMAGE_1'].iloc[1] < 0.1:
    #     return False
    # # First check flags.
    # if many_to_one_subset['FLAGS_1'].iloc[0] > 7 or many_to_one_subset['FLAGS_1'].iloc[0] > 7:
    #     return False
    # ocoord = (many_to_one_subset['X_IMAGE_DBL_2'].iloc[0], many_to_one_subset['Y_IMAGE_DBL_2'].iloc[0])
    # dcoord1 = (many_to_one_subset['X_IMAGE_DBL_1'].iloc[0], many_to_one_subset['Y_IMAGE_DBL_1'].iloc[0])
    # assert len(orow) == 1
    # assert len(drow) == 1
    ocoord = tuple(orow[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
    dcoord1 = tuple(drow[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
    dist1 = math.dist(ocoord, dcoord1)
    sigma_j = orow['FWHM_IMAGE']
    if np.isclose(sigma_j, 0.0):
        return False
    # sigma_i1 = many_to_one_subset['FWHM_IMAGE_1'].iloc[0]
    # sigma_i2 = many_to_one_subset['FWHM_IMAGE_1'].iloc[1]
    # We approximate the Gaussian-convolved FWHM of the deconvolved with the original source's FWHM for simplicity.
    sigma_i1, sigma_i2 = sigma_j, sigma_j
    dist_eff1 = dist1 / (norm_factor * (sigma_j + sigma_i1))
    condition1 = dist_eff1 < 1
    # This condition is more stricter to ensure a purer deblended sample.
    condition2 = (
        orow['ELLIPTICITY'] > drow['ELLIPTICITY'] - 0.05
    )
    return condition1 and condition2

def find_closest_orig_to_deconv(orig_df, deconv_row):
    min_d = np.Inf
    for i, row in enumerate(orig_df.iterrows()):
        d = math.dist(
            tuple(row[1][['X_IMAGE_DBL', 'Y_IMAGE_DBL']]),
            tuple(deconv_row[['X_IMAGE_DBL', 'Y_IMAGE_DBL']])
        )
        if d < min_d:
            min_d = d
            min_row_number = i

    return orig_df.iloc[i]

In [ ]:
def mag_to_flux(m):
    return 10 ** (0.4 * (magzp - m))

def flux_to_mag(flux):
    m = magzp - 2.5 * np.log10(flux) if flux > 0 else 99
    return m

def reduce_C_function(C):
    C = np.array(C)
    magzp = image_header['MAGZP']

    flux_sum = np.sum([mag_to_flux(c) for c in C])
    return flux_to_mag(flux_sum)

Checking for why sources in the deconvolved brighter than the limiting mag of deepref are not found in deepref when they actually should. One possible reason is deconvolution may have deblended such that deepref source was matched with one but the other deconv component was farther than the threshold but still in proximity.

But in fact, such cases were almost none.

In [ ]:
# after first review.
unmatches_deconvolved_cut_astrophysical = unmatches_deconvolved_cut[unmatches_deconvolved_cut['FLAGS'] <= 7]
# see https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=terms-modified-z-score
# 1.22 is the median of deconv sources from the 1-1 match for 626-r and 0.25 is the MAD.
robust_zscores = (unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'] - 1.22) / (1.486 * 0.25)
robust_zscores_only = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3)]
print(f'No. of unreliable deconv sources using zscore criteria = {len(robust_zscores_only)}/{len(unmatches_deconvolved_cut_astrophysical)}')
robust_zscores_fwhmellzero = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.)))]
print(f'No. of unreliable deconv sources using zscore criteria and fwhm/ell near zero and ell < 0.6 criteria = {len(robust_zscores_fwhmellzero)}/{len(unmatches_deconvolved_cut_astrophysical)}')

unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell = unmatches_deconvolved_cut_astrophysical[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.))) | (unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'] >= 0.6)]
unmatches_deconvolved_cut_astrophysical_final = unmatches_deconvolved_cut_astrophysical[(robust_zscores > -3) & (robust_zscores < 3) & (~((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.)))) & (unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'] < 0.6)]

print(f'No. of final unmatched deconv DUBIOUS sources = {len(unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell)}')
print(f'No. of final unmatched deconv astrophysical sources = {len(unmatches_deconvolved_cut_astrophysical_final)}')

assert len(unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell) + len(unmatches_deconvolved_cut_astrophysical_final) == len(unmatches_deconvolved_cut_astrophysical)
unmatches_deconvolved_cut_astrophysical_final.columns

In [ ]:
unmatches_deconvolved_cut_astrophysical_final['SUBDIV_NUMBER'].value_counts()

In [ ]:
visualize_cutouts = True

# after first review.
unmatches_deconvolved_cut_astrophysical = unmatches_deconvolved_cut[unmatches_deconvolved_cut['FLAGS'] <= 7]
# see https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=terms-modified-z-score
# 1.22 is the median of deconv sources from the 1-1 match for 626-r and 0.25 is the MAD.
robust_zscores = (unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'] - 1.22) / (1.486 * 0.25)
robust_zscores_only = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3)]
print(f'No. of unreliable deconv sources using zscore criteria = {len(robust_zscores_only)}/{len(unmatches_deconvolved_cut_astrophysical)}')
robust_zscores_fwhmellzero = robust_zscores[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.)))]
print(f'No. of unreliable deconv sources using zscore criteria and fwhm/ell near zero and ell < 0.6 criteria = {len(robust_zscores_fwhmellzero)}/{len(unmatches_deconvolved_cut_astrophysical)}')

unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell = unmatches_deconvolved_cut_astrophysical[(robust_zscores < -3) | (robust_zscores > 3) | ((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.))) | (unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'] >= 0.6)]
unmatches_deconvolved_cut_astrophysical_final = unmatches_deconvolved_cut_astrophysical[(robust_zscores > -3) & (robust_zscores < 3) & (~((np.isclose(unmatches_deconvolved_cut_astrophysical['FWHM_IMAGE'], 0.)) & (np.isclose(unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'], 0.)))) & (unmatches_deconvolved_cut_astrophysical['ELLIPTICITY'] < 0.6)]

print(f'No. of final unmatched deconv DUBIOUS sources = {len(unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell)}')
print(f'No. of final unmatched deconv astrophysical sources = {len(unmatches_deconvolved_cut_astrophysical_final)}')

assert len(unmatches_deconvolved_cut_astrophysical_zscore_fwhmellzero_ell) + len(unmatches_deconvolved_cut_astrophysical_final) == len(unmatches_deconvolved_cut_astrophysical)

################ Search for blended original sources ################
# NOTE: ALSO NOTE THAT WE USE ocat INSTEAD ocat_cut.
orig_img = '/content/ztf_000626_zr_c12_q2_deeprefimg_align1.fits'
deconv_img = '/content/deconvolved_subdiv_ztf_20221120493183_000626_zr_c12_o_q2_sciimg_align1.fits'
rect_d = 7
count = 0

from astropy.wcs import WCS
w1 = WCS(fits.open('/content/deconvolved_subdiv_ztf_20221120493183_000626_zr_c12_o_q2_sciimg.fits')[0].header)
w2 = WCS(fits.open('/content/ztf_000626_zr_c12_q2_refimg_align1.fits')[0].header)

most_closest_orig_sources = []
deconv_sources = []

from regions import CircleSkyRegion, CirclePixelRegion
from regions import PixCoord
from astropy.coordinates import Angle, SkyCoord

import astropy.units as u

already_done_orig_sources = []
for row in unmatches_deconvolved_cut_astrophysical_final.iterrows():
    # We need to use sky coordinates for orig/ref.
    search_radius = Angle(rect_d/3600, 'deg')
    _c = SkyCoord(ra=row[1]['X_WORLD']*u.deg, dec=row[1]['Y_WORLD']*u.deg)
    sky_region = CircleSkyRegion(_c, search_radius)
    # See https://astropy-regions.readthedocs.io/en/stable/contains.html
    _oc = SkyCoord(ra=[o*u.deg for o in ocat['X_WORLD']], dec=[o*u.deg for o in ocat['Y_WORLD']])
    closest_orig_source = ocat[sky_region.contains(_oc, wcs=w1)]
    # Can use pixel coords since both here are deconv.
    closest_deconv_source = dcat[(dcat['X_IMAGE_DBL'] >= row[1]['X_IMAGE_DBL']-rect_d) & (dcat['X_IMAGE_DBL'] <= row[1]['X_IMAGE_DBL']+rect_d) & (dcat['Y_IMAGE_DBL'] >= row[1]['Y_IMAGE_DBL']-rect_d) & (dcat['Y_IMAGE_DBL'] <= row[1]['Y_IMAGE_DBL']+rect_d)]
    closest_deconv_source = closest_deconv_source[(closest_deconv_source['X_IMAGE_DBL'] != row[1]['X_IMAGE_DBL']) & (closest_deconv_source['Y_IMAGE_DBL'] != row[1]['Y_IMAGE_DBL'])]

    print(len(closest_orig_source), len(closest_deconv_source))

    # Only when this deconv source has an original counterpart and also another deconvolved source, it might be a deblend.
    if len(closest_orig_source) > 0 and len(closest_deconv_source) > 0:
        if len(closest_orig_source) > 1:
            most_closest_orig_source = find_closest_orig_to_deconv(closest_orig_source, row[1])
        elif len(closest_orig_source) == 1:
            closest_orig_source = closest_orig_source.iloc[0]
            most_closest_orig_source = closest_orig_source

        # We are only interested in orig converted into two deconv sources.
        # One is the one on which are looping through unmatch_dcat and the other is in `closest_deconv_source`.
        if len(closest_deconv_source) != 1:
            continue

        ################# THESE ARE LESS IMPORTANT CONDITIONS BUT STILL USEFUL FOR OBTAINING PURITY #################
        if closest_deconv_source['ELLIPTICITY'].iloc[0] >= 0.6 or row[1]['ELLIPTICITY'] >= 0.6:  # we only want deconvolved sources with ellipticity < 0.6.
            continue

        if np.isclose(closest_deconv_source['FWHM_IMAGE'].iloc[0], 0) or np.isclose(row[1]['FWHM_IMAGE'], 0):
            continue

        if most_closest_orig_source['ELLIPTICITY'] < 0.1:
            continue
        #############################################################################################################

        print('reached ckpt 1')

        if most_closest_orig_source['FLAGS'] > 7:
            continue

        if check_particular_flag(most_closest_orig_source['FLAGS'], nth_bit_from_right=1):  # deblending flag for the orig must not be set.
            continue

        if most_closest_orig_source['NUMBER'] in already_done_orig_sources:
            continue

        print('reached ckpt 2')

        # We use the most closest original source for deblending check.
        orig_blended_or_not = blended_or_not(most_closest_orig_source, row[1], norm_factor=1.)
        # We are only interested when the orig source is blended.
        if orig_blended_or_not:
            count += 1
            already_done_orig_sources.append(most_closest_orig_source['NUMBER'])
            # if count == 40:
            #     break
            dist_bet_deconv_sources = math.dist(
                (row[1]['X_IMAGE_DBL'], row[1]['Y_IMAGE_DBL']),
                (closest_deconv_source['X_IMAGE_DBL'].iloc[0], closest_deconv_source['Y_IMAGE_DBL'].iloc[0])
            )
            if dist_bet_deconv_sources > 2.5:
                continue
            if dist_bet_deconv_sources < 2.5:
                print('GOT A SOURCE WITH < 2.5 PIX SEPARATION!!!')
            most_closest_orig_sources.append(most_closest_orig_source)
            deconv_sources.append((
                [closest_deconv_source['ELLIPTICITY'].iloc[0], row[1]['ELLIPTICITY']],
                [closest_deconv_source['MAG_ISO'].iloc[0], row[1]['MAG_ISO']],
                dist_bet_deconv_sources
            ))

            ##### Comment from this line onwards if only statistics are needed and no visualization ######
            # print(count)
            if visualize_cutouts:
                ocoord = (most_closest_orig_source['X_IMAGE_DBL'], most_closest_orig_source['Y_IMAGE_DBL'])

                dcut = Cutout2D(deconv_img, ocoord, size=30)
                ocut = Cutout2D(orig_img, ocoord, size=30)
                onorm = ImageNormalize(ocut.data, interval=PercentileInterval(99.5),
                                stretch=SqrtStretch())
                dnorm = ImageNormalize(dcut.data, interval=PercentileInterval(99.5),
                                stretch=SqrtStretch())
                fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
                ax[0].imshow(ocut.data, origin='lower', interpolation='nearest', cmap='inferno', norm=onorm)
                ax[1].imshow(dcut.data, origin='lower', interpolation='nearest', cmap='inferno', norm=dnorm)

                text_x, text_y = .1, .95
                ax[0].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(most_closest_orig_source["FWHM_IMAGE"], 2)} pix, {np.round(most_closest_orig_source["ELLIPTICITY"], 2)}, {np.round(most_closest_orig_source["MAG_ISO"], 2)}, {int(most_closest_orig_source["FLAGS"])}', ha='left', va='top', transform=ax[0].transAxes, color='white', fontsize=axeslabelsize, weight='bold', family='sans-serif', linespacing=1.5)
                ax[1].text(text_x, text_y, f'FWHM, ellipticity, mag, FLAG\n{np.round(row[1]["FWHM_IMAGE"], 2)} pix, {np.round(row[1]["ELLIPTICITY"], 2)}, {np.round(row[1]["MAG_ISO"], 2)}, {int(row[1]["FLAGS"])}\n{np.round(closest_deconv_source["FWHM_IMAGE"].iloc[0], 2)} pix, {np.round(closest_deconv_source["ELLIPTICITY"].iloc[0], 2)}, {np.round(closest_deconv_source["MAG_ISO"].iloc[0], 2)}, {int(closest_deconv_source["FLAGS"].iloc[0])}', ha='left', va='top', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize, weight='bold', family='sans-serif', linespacing=1.5)
                ax[1].text(0.3, 0.3, f'dist = {dist_bet_deconv_sources:.2f} pix', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize, weight='bold', family='sans-serif', linespacing=1.5)
                total_mag = flux_to_mag(mag_to_flux(row[1]["MAG_ISO"]) + mag_to_flux(closest_deconv_source["MAG_ISO"].iloc[0]))
                ax[1].text(0.25, 0.2, f'total mag = {total_mag:.2f}', transform=ax[1].transAxes, color='white', fontsize=axeslabelsize, weight='bold', family='sans-serif', linespacing=1.5)

                oo = ocut.to_cutout_position(ocoord)
                dd = dcut.to_cutout_position((row[1]['X_IMAGE_DBL'], row[1]['Y_IMAGE_DBL']))
                dd2 = dcut.to_cutout_position((closest_deconv_source['X_IMAGE_DBL'].iloc[0], closest_deconv_source['Y_IMAGE_DBL'].iloc[0]))

                mul_factor = 6
                e = Ellipse(xy=oo,
                    width=mul_factor*most_closest_orig_source['A_IMAGE'],
                    height=mul_factor*most_closest_orig_source['B_IMAGE'],
                    angle=most_closest_orig_source['THETA_IMAGE'] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[0].add_artist(e)

                mul_factor = 12
                e = Ellipse(xy=dd,
                    width=mul_factor*row[1]['A_IMAGE'],
                    height=mul_factor*row[1]['B_IMAGE'],
                    angle=row[1]['THETA_IMAGE'] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[1].add_artist(e)

                mul_factor = 12
                e = Ellipse(xy=dd2,
                    width=mul_factor*closest_deconv_source['A_IMAGE'].iloc[0],
                    height=mul_factor*closest_deconv_source['B_IMAGE'].iloc[0],
                    angle=closest_deconv_source['THETA_IMAGE'].iloc[0] * 180. / np.pi,
                    linewidth=1.2
                )
                e.set_facecolor('none')
                e.set_edgecolor('#a8ddb5')
                ax[1].add_artist(e)

                for a in ax:
                    a.set_xticks([])
                    a.set_yticks([])

                # plt.savefig(f'deblend_example_{title}_{count}.pdf', bbox_inches='tight', dpi=200, format='pdf')
                # plt.clf()

                plt.show()
